# Mistral-7B DeepSeek-R1 Style Medical Chain-of-Thought (CoT) Fine-Tuning

This notebook fine-tunes **Mistral-7B-Instruct-v0.3** using **4-bit QLoRA** on a **Kaggle P100 GPU (16GB VRAM)** for clinical diagnosis and reasoning using the [`theguywithblacktie/medical-cot`](https://huggingface.co/datasets/theguywithblacktie/medical-cot) dataset.

### Supported Experiment Modes:
- `cot`: Standard SFT with DeepSeek-R1-style `<think>...</think>` reflection and completion masking on `[/INST]`.
- `direct`: Baseline SFT directly generating diagnosis without reasoning scratchpad.
- `weighted_cot`: Progressive per-token loss weighting (`<think>`: 0.25, `reasoning`: 0.50, `</think>`: 0.75, `answer`: 1.00).

In [ ]:
# [CELL 1] Clone or Pull Repository from GitHub (Branch: cot)
import os
import sys

repo_dir = "/kaggle/working/peft-recipes-lab"
repo_url = "https://github.com/theGuyWithBlackTie/peft-recipes-lab.git"
branch = "cot"

if not os.path.exists(repo_dir):
    print(f"Cloning {repo_url} (branch: {branch}) into {repo_dir}...")
    !git clone -b {branch} {repo_url} {repo_dir}
else:
    print(f"Updating existing repo in {repo_dir}...")
    !cd {repo_dir} && git fetch origin && git checkout {branch} && git pull origin {branch}

# Ensure project root is in sys.path
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Change working directory to project root
os.chdir(repo_dir)
print("Current working directory:", os.getcwd())

In [ ]:
# [CELL 2] Install / Verify Kaggle Dependencies
!pip install -q --upgrade pip
!pip install -q transformers peft trl bitsandbytes datasets accelerate pyyaml huggingface_hub

In [ ]:
# [CELL 3] Authenticate with Hugging Face Hub
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception as e:
        print("Kaggle secret HF_TOKEN not found:", e)

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("✅ Successfully authenticated with Hugging Face Hub.")
else:
    print("⚠️ No HF_TOKEN found. Public datasets/models will work; pushing to hub will fail.")

In [ ]:
# [CELL 4] Load Configuration and Select Experiment Mode
import yaml

config_path = "lora/config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Select Experiment Mode: 'cot', 'direct', or 'weighted_cot'
EXPERIMENT_MODE = "cot"  # Change to 'direct' or 'weighted_cot' as needed
config["experiment_mode"] = EXPERIMENT_MODE

# Ensure P100 memory settings (16GB VRAM)
config["batch_size"] = 2
config["gradient_accumulation_steps"] = 8
config["max_seq_length"] = 2048
config["use_fp16"] = True
config["use_bf16"] = False
config["use_4bit"] = True
config["output_dir"] = f"./mistral-{EXPERIMENT_MODE}-lora-outputs"

if EXPERIMENT_MODE == "weighted_cot":
    config["loss_weighting"]["enabled"] = True

print(f"Configured Experiment Mode: {config['experiment_mode']}")
print(f"Target Output Directory: {config['output_dir']}")
print(f"Dataset Path: {config['dataset_path']}")

In [ ]:
# [CELL 5] Build Model & Tokenizer using ModelBuilder
from lora.src.model_builder import ModelBuilder
import torch

print("Building 4-bit QLoRA model for Mistral-7B...")
model_builder = ModelBuilder(config)
model, tokenizer = model_builder.build()

print("\n--- Model Memory Footprint ---")
print(f"Allocated VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved VRAM:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
# [CELL 6] Load and Prepare Dataset using DataLoader
from lora.src.data_loader import DataLoader

print("Loading dataset...")
data_loader = DataLoader(config, tokenizer)
train_dataset, eval_dataset = data_loader.load_and_format_data()

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples:  {len(eval_dataset)}")
print("\nSample formatted text:")
print(train_dataset[0]["text"][:400] + "...")

In [ ]:
# [CELL 7] Initialize Trainer (Dynamic SFT or Progressive Weighted Loss)
from lora.src.trainer import Trainer

print(f"Setting up Trainer for mode: {config['experiment_mode']}...")
trainer_engine = Trainer(
    config=config,
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

trainer = trainer_engine.setup_trainer()
print("✅ Trainer successfully initialized.")

In [ ]:
# [CELL 8] Execute Fine-Tuning
print("Starting training on GPU...")
train_result = trainer.train()

print("\nTraining complete! Results:")
print(train_result)

In [ ]:
# [CELL 9] Save LoRA Adapter and Push to Hugging Face Hub
print(f"Saving trained adapter to {config['output_dir']}...")
trainer.save_model(config["output_dir"])
tokenizer.save_pretrained(config["output_dir"])

if config.get("push_to_hub", False):
    print(f"Pushing adapter to Hub: {config['hub_model_id']}...")
    trainer.push_to_hub()
    print("✅ Model pushed to Hugging Face Hub!")

In [ ]:
# [CELL 10] Test Clinical Diagnostic Inference with <think> Traces
from lora.src.inference import InferenceEngine

print("Running test clinical inference on the fine-tuned model...")
infer_engine = InferenceEngine(model=model, tokenizer=tokenizer, config=config)

test_prompt = (
    "[INST] Below is an instruction that describes a task, paired with an input that provides further context.\n"
    "Write a response that appropriately completes the request.\n"
    "Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.\n\n"
    "### Instruction:\n"
    "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.\n"
    "Please answer the following medical question.\n\n"
    "### Question:\n"
    "A 45-year-old male presents with sudden-onset crushing chest pain radiating to his left jaw and diaphoresis. "
    "ECG demonstrates 3mm ST-segment elevation in leads V1-V4. What is the immediate definitive intervention? [/INST]"
)

generated_output = infer_engine.generate(test_prompt)
print("\n--- Generated Diagnostic Output ---\n")
print(generated_output)